# AgriShift — 02. Data Cleaning & Validation

**Project:** Agricultural Diversification & Crop Pattern Intelligence in India  
**SDG:** 2 — Zero Hunger

## Purpose
Create a cleaned copy of the original crop-production dataset. The raw CSV will remain unchanged.

### Decisions from Notebook 01
1. Standardize whitespace in text columns.
2. Ensure year, area and production are numeric.
3. Preserve missing `Production` values; do not invent replacements.
4. Preserve zero-production rows for investigation.
5. Retain 2015 in the master cleaned data, but do not use it for full-India comparisons because only Odisha and Sikkim appear in that year.
6. Do not silently delete rows or duplicates.

**Output:** `data/processed/crop_production_cleaned.csv`

## 1. Import libraries

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

## 2. Locate and load the original dataset

Expected location: `data/raw/crop_production.csv`.

In [2]:
DATA_RELATIVE_PATH = Path("data") / "raw" / "crop_production.csv"

PROJECT_ROOT = next(
    (
        folder
        for folder in (Path.cwd(), *Path.cwd().parents)
        if (folder / DATA_RELATIVE_PATH).is_file()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find data/raw/crop_production.csv. "
        "Open the AgriShift repository in VS Code and check the dataset location."
    )

RAW_DATA_PATH = PROJECT_ROOT / DATA_RELATIVE_PATH
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

df_raw = pd.read_csv(RAW_DATA_PATH)
df_clean = df_raw.copy(deep=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Original rows: {len(df_raw):,}")
print(f"Original columns: {df_raw.shape[1]}")

Project root: c:\Users\ajadi\OneDrive\Desktop\CSD\Exploration\AgriShift
Original rows: 246,091
Original columns: 7


## 3. Validate the expected columns

In [3]:
expected_columns = [
    "State_Name", "District_Name", "Crop_Year",
    "Season", "Crop", "Area", "Production"
]

missing_columns = [col for col in expected_columns if col not in df_clean.columns]
if missing_columns:
    raise ValueError(f"Required columns are missing: {missing_columns}")

df_clean = df_clean[expected_columns].copy()
display(df_clean.head())

,State_Name,District_Name,Crop_Year,Season,Crop,Area,Production
0,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Arecanut,"1,254.00","2,000.00"
1,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Other Kharif pulses,2.00,1.00
2,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Rice,102.00,321.00
3,Andaman and Nicobar Islands,NICOBARS,2000,Whole Year,Banana,176.00,641.00
4,Andaman and Nicobar Islands,NICOBARS,2000,Whole Year,Cashewnut,720.00,165.00


## 4. Standardize text fields

The first notebook found trailing spaces in labels such as `Kharif     `. We remove leading/trailing spaces and collapse repeated whitespace, while preserving the original wording and capitalization.

In [4]:
text_columns = ["State_Name", "District_Name", "Season", "Crop"]
text_log = []

for col in text_columns:
    before = df_clean[col].astype("string")
    after = before.str.replace(r"\s+", " ", regex=True).str.strip()
    changed = int((before.fillna("<NA>") != after.fillna("<NA>")).sum())

    text_log.append({
        "column": col,
        "values_changed": changed,
        "unique_values_after_cleaning": after.nunique(dropna=True)
    })
    df_clean[col] = after

display(pd.DataFrame(text_log))

print("Cleaned season labels:")
for value in sorted(df_clean["Season"].dropna().unique()):
    print(repr(value))

,column,values_changed,unique_values_after_cleaning
0,State_Name,7283,33
1,District_Name,0,646
2,Season,246091,6
3,Crop,5176,124


Cleaned season labels:
'Autumn'
'Kharif'
'Rabi'
'Summer'
'Whole Year'
'Winter'


## 5. Convert numeric columns safely

In [5]:
numeric_columns = ["Crop_Year", "Area", "Production"]
numeric_log = []

for col in numeric_columns:
    original_non_missing = df_clean[col].notna()
    converted = pd.to_numeric(df_clean[col], errors="coerce")
    failures = int((original_non_missing & converted.isna()).sum())

    numeric_log.append({
        "column": col,
        "conversion_failures": failures,
        "missing_after_conversion": int(converted.isna().sum())
    })
    df_clean[col] = converted

if df_clean["Crop_Year"].isna().any():
    raise ValueError("Crop_Year contains missing or non-numeric values.")

df_clean["Crop_Year"] = df_clean["Crop_Year"].astype("int64")
display(pd.DataFrame(numeric_log))

,column,conversion_failures,missing_after_conversion
0,Crop_Year,0,0
1,Area,0,0
2,Production,0,3730


## 6. Check missing and blank identification fields

In [6]:
key_columns = ["State_Name", "District_Name", "Crop_Year", "Season", "Crop"]

print("Missing values in identification fields:")
display(df_clean[key_columns].isna().sum().to_frame("missing_count"))

blank_text = {
    col: int(df_clean[col].astype("string").str.strip().eq("").sum())
    for col in ["State_Name", "District_Name", "Season", "Crop"]
}
print("Blank text values:")
display(pd.Series(blank_text, name="blank_count").to_frame())

if df_clean[key_columns].isna().any().any() or sum(blank_text.values()) > 0:
    print("Review affected records before modelling. They are not deleted automatically.")
else:
    print("All record-identification fields are populated.")

Missing values in identification fields:


,missing_count
State_Name,0
District_Name,0
Crop_Year,0
Season,0
Crop,0


Blank text values:


,blank_count
State_Name,0
District_Name,0
Season,0
Crop,0


All record-identification fields are populated.


## 7. Review missing and zero production values

The first notebook found 3,730 missing production values and 3,523 zero-production values.

Area is the primary variable for our diversification analysis. We will not delete a row simply because its production value is missing. Missing and zero production values are preserved for now.

In [7]:
production_review = pd.Series({
    "Missing Production": int(df_clean["Production"].isna().sum()),
    "Zero Production": int(df_clean["Production"].eq(0).sum()),
    "Negative Production": int(df_clean["Production"].lt(0).sum()),
    "Missing Area": int(df_clean["Area"].isna().sum()),
    "Zero Area": int(df_clean["Area"].eq(0).sum()),
    "Negative Area": int(df_clean["Area"].lt(0).sum()),
}, name="record_count").to_frame()

display(production_review)
print("No production values have been imputed or replaced.")

,record_count
Missing Production,3730
Zero Production,3523
Negative Production,0
Missing Area,0
Zero Area,0
Negative Area,0


No production values have been imputed or replaced.


## 8. Check duplicate records after text standardization

Whitespace changes can cause previously different-looking keys to become identical. Check duplicates again; do not remove them automatically.

In [8]:
record_key = ["State_Name", "District_Name", "Crop_Year", "Season", "Crop"]
duplicate_mask = df_clean.duplicated(subset=record_key, keep=False)
duplicate_count = int(df_clean.duplicated(subset=record_key).sum())

print(f"Duplicate logical records: {duplicate_count:,}")

if duplicate_count:
    display(df_clean.loc[duplicate_mask].sort_values(record_key).head(50))
    print("Investigate these records before deciding whether to remove any.")
else:
    print("No duplicate logical records were introduced by text standardization.")

Duplicate logical records: 0
No duplicate logical records were introduced by text standardization.


## 9. Validate area and production ranges

In [9]:
range_checks = pd.Series({
    "Area below zero": int(df_clean["Area"].lt(0).sum()),
    "Area equal to zero": int(df_clean["Area"].eq(0).sum()),
    "Production below zero": int(df_clean["Production"].lt(0).sum()),
    "Production equal to zero": int(df_clean["Production"].eq(0).sum()),
    "Area missing": int(df_clean["Area"].isna().sum()),
}, name="record_count").to_frame()

display(range_checks)

if df_clean["Area"].isna().any() or df_clean["Area"].lt(0).any():
    print("Area has missing or negative values. Review before feature engineering.")
else:
    print("Area has no missing or negative values.")

,record_count
Area below zero,0
Area equal to zero,0
Production below zero,0
Production equal to zero,3523
Area missing,0


Area has no missing or negative values.


## 10. Review year-wise data coverage

The 2015 data contains only Odisha and Sikkim. Retain it in the cleaned master file for traceability, but exclude it from comparisons that require full-India coverage.

This does not prove that every year from 1997–2014 is complete. We will review state coverage before making national trend claims.

In [10]:
year_summary = (
    df_clean.groupby("Crop_Year")
    .agg(
        record_count=("Crop_Year", "size"),
        states_present=("State_Name", "nunique"),
        districts_present=("District_Name", "nunique")
    )
    .sort_index()
)

display(year_summary)

latest_year = int(df_clean["Crop_Year"].max())
latest_states = sorted(
    df_clean.loc[df_clean["Crop_Year"].eq(latest_year), "State_Name"].dropna().unique()
)
print(f"Latest year: {latest_year}")
print(f"States present in latest year: {latest_states}")

,record_count,states_present,districts_present
Crop_Year,,,
1997,8899,23,473
1998,11533,27,495
1999,12515,28,514
2000,13658,31,543
2001,13361,31,542
2002,16671,32,565
2003,17287,32,580
2004,14117,33,585
2005,13799,33,590


Latest year: 2015
States present in latest year: ['Odisha', 'Sikkim']


## 11. Final validation before saving

In [11]:
assert len(df_clean) == len(df_raw), "Row count changed unexpectedly."
assert df_clean.columns.tolist() == expected_columns, "Column structure changed."
assert df_clean["Crop_Year"].notna().all(), "Crop_Year has missing values."
assert not df_clean["Area"].lt(0).any(), "Negative Area values remain."

print("Final validation passed.")
print(f"Rows retained: {len(df_clean):,}")
print(f"Columns retained: {df_clean.shape[1]}")
print(f"Missing Production retained: {df_clean['Production'].isna().sum():,}")
print(f"Zero Production retained: {df_clean['Production'].eq(0).sum():,}")

Final validation passed.
Rows retained: 246,091
Columns retained: 7
Missing Production retained: 3,730
Zero Production retained: 3,523


## 12. Save the cleaned dataset

In [12]:
OUTPUT_PATH = PROCESSED_DIR / "crop_production_cleaned.csv"
df_clean.to_csv(OUTPUT_PATH, index=False)

print(f"Cleaned dataset saved to: {OUTPUT_PATH}")
print(f"File size: {OUTPUT_PATH.stat().st_size / (1024 * 1024):.2f} MB")

Cleaned dataset saved to: c:\Users\ajadi\OneDrive\Desktop\CSD\Exploration\AgriShift\data\processed\crop_production_cleaned.csv
File size: 13.05 MB


## 13. Verify the saved file

In [13]:
df_check = pd.read_csv(OUTPUT_PATH)

print(f"Rows: {len(df_check):,}")
print(f"Columns: {df_check.shape[1]}")
print(f"Column names match: {df_check.columns.tolist() == expected_columns}")
print(f"Shape matches: {df_check.shape == df_clean.shape}")

assert df_check.shape == df_clean.shape
assert df_check.columns.tolist() == expected_columns

display(df_check.head())

Rows: 246,091
Columns: 7
Column names match: True
Shape matches: True


,State_Name,District_Name,Crop_Year,Season,Crop,Area,Production
0,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Arecanut,"1,254.00","2,000.00"
1,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Other Kharif pulses,2.00,1.00
2,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Rice,102.00,321.00
3,Andaman and Nicobar Islands,NICOBARS,2000,Whole Year,Banana,176.00,641.00
4,Andaman and Nicobar Islands,NICOBARS,2000,Whole Year,Cashewnut,720.00,165.00


## 14. Stage summary

### Decisions applied
- Standardized whitespace in text fields.
- Converted year, area and production to numeric types.
- Preserved missing production values.
- Preserved zero-production records.
- Retained all source rows, including 2015.
- Saved a separate cleaned CSV; the original raw file was not modified.

### Questions for later stages
- Are years before 2015 sufficiently comparable for India-wide analysis?
- Do district names remain consistent over time?
- Are any crop labels aliases for the same crop?
- What units are used for area and production in the original source metadata?

**Stage output:** `data/processed/crop_production_cleaned.csv`

No statistical imputation or row deletion has been performed.